# Species Richness

In [ ]:
### Read in latest season's data from CSV

INPUT_DATA_PATH = 'data/input/'

import pandas as pd
df = pd.read_csv(INPUT_DATA_PATH + 'Underconstruction Benthic DECFEB.csv')
df.head()

In [ ]:
## Filter out columns we don't need for species richness
df = df[['Survey_ID','Site', 'Species', 'Total']]
# NOTE: Leave filtering by Survey Status for later - I know by inspection that there are only "complete" (1) surveys

## Filter by specific site for now
CHOSEN_SITE = 'Mojon MPA'
filtered_df = df[df['Site'] == CHOSEN_SITE]
print("Number of rows for {}: {}".format(CHOSEN_SITE, len(filtered_df)))
filtered_df.head()

In [ ]:
## Calculate number of surveys this season
num_surveys = filtered_df['Survey_ID'].nunique()
print("Number of surveys for {}: {}".format(CHOSEN_SITE, num_surveys))

# Remove Survey ID - no longer needed
filtered_df = filtered_df.drop(columns=['Survey_ID'])

In [ ]:
## Calculate species richness (number of unique species surveyed) averaged (over surveys)
# Aggregate by species to get total abundance of each species across all surveys
species_counts = filtered_df.groupby('Species')['Total'].sum().reset_index()
species_richness = species_counts['Species'].nunique()

# Calculate average species richness per survey
average_species_richness = species_richness / num_surveys
print("Total/Average Species Richness for {}: {}, {:.2f}".format(CHOSEN_SITE, species_richness, average_species_richness))

In [ ]:
## Calculate species richness for important indicator species

# Giant Clams
gc_species_categories = ["Bivalves - Giant Clam", "Bivalves - Boring Giant Clam"]
giant_clam_counts = filtered_df[filtered_df['Species'].isin(gc_species_categories)].groupby('Species')['Total'].sum().reset_index()
giant_clam_abundance = giant_clam_counts['Total'].sum()
average_giant_clam_abundance = giant_clam_abundance / num_surveys
giant_clam_richness = giant_clam_counts['Species'].nunique()
average_giant_clam_richness = giant_clam_richness / num_surveys
print("Total/Average Giant Clam Abundance {}: {}, {:.2f}".format(CHOSEN_SITE, giant_clam_abundance, average_giant_clam_abundance))
print("Total/Average Giant Clam Species Richness {}: {}, {:.2f}".format(CHOSEN_SITE, giant_clam_richness, average_giant_clam_richness))
print("-----------------------------------------------------------------")

# Sea Urchins
urchin_species_categories = ["Sea Urchins - Diadema", "Sea Urchins - Rock Boring", "Sea Urchins - Collector"]
urchin_counts = filtered_df[filtered_df['Species'].isin(urchin_species_categories)].groupby('Species')['Total'].sum().reset_index()
urchin_abundance = urchin_counts['Total'].sum()
average_urchin_abundance = urchin_abundance / num_surveys
urchin_richness = urchin_counts['Species'].nunique()
average_urchin_richness = urchin_richness / num_surveys
print("Total/Average Sea Urchin Abundance {}: {}, {:.2f}".format(CHOSEN_SITE, urchin_abundance, average_urchin_abundance))
print("Total/Average Sea Urchin Species Richness {}: {}, {:.2f}".format(CHOSEN_SITE, urchin_richness, average_urchin_richness))
print("-----------------------------------------------------------------")

# Sea Cucumbers
cucumber_species_categories = ["Sea Cucumbers - Pinkfish",
                                    "Sea Cucumbers - Black Spotted",
                                    "Sea Cucumbers - Other",
                                    "Sea Cucumbers - Leopard",
                                    "Sea Cucumbers - Amberfish",
                                    "Sea Cucumbers - Volcano",
                                    "Sea Cucumbers - Golden Sandfish",
                                    "Sea Cucumbers - Magnum"
                            ]
cucumber_counts = filtered_df[filtered_df['Species'].isin(cucumber_species_categories)].groupby('Species')['Total'].sum().reset_index()
cucumber_abundance = cucumber_counts['Total'].sum()
average_cucumber_abundance = cucumber_abundance / num_surveys
cucumber_richness = cucumber_counts['Species'].nunique()
average_cucumber_richness = cucumber_richness / num_surveys
print("Total/Average Sea Cucumber Abundance {}: {}, {:.2f}".format(CHOSEN_SITE, cucumber_abundance, average_cucumber_abundance))
print("Total/Average Sea Cucumber Species Richness {}: {}, {:.2f}".format(CHOSEN_SITE, cucumber_richness, average_cucumber_richness))

In [ ]:
### Compute species richness (total and average) for all sites

## Filter out columns we don't need for species richness
df = df[['Survey_ID','Site', 'Species', 'Total']]
# NOTE: Leave filtering by Survey Status for later - I know by inspection that there are only "complete" (1) surveys

## Get complete list of sites
sites = df['Site'].unique()

# Store results for each site
results = []

for site in sites:
    filtered_df = df[df['Site'] == site]

    ## Calculate number of surveys this season
    num_surveys = filtered_df['Survey_ID'].nunique()
    filtered_df = filtered_df.drop(columns=['Survey_ID']) # Remove Survey ID - no longer needed

    ## Calculate species richness (number of unique species surveyed) total and averaged over surveys
    species_counts = filtered_df.groupby('Species')['Total'].sum().reset_index()
    species_richness = species_counts['Species'].nunique()

    # Calculate average species richness per survey
    average_species_richness = round(species_richness / num_surveys, 2)
    
    # Store results
    results.append({
        'Site': site,
        'Species Richness': species_richness,
        'Average Species Richness': average_species_richness,
        'Number of Surveys': num_surveys
    })

results_df = pd.DataFrame(results)
print("\nSpecies Richness Results:")
results_df

In [ ]:
### Visualize Species Richness Results

import matplotlib.pyplot as plt
import numpy as np

# Create a figure with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# 1. Combined Bar Chart - Total and Average Species Richness by Site
ax1 = axes[0]
ax1_twin = ax1.twinx()  # Create a second y-axis

sorted_df = results_df.sort_values('Species Richness', ascending=False)

x = np.arange(len(sorted_df))
width = 0.35

# Plot total species richness on the primary y-axis
bars1 = ax1.bar(x - width/2, sorted_df['Species Richness'], width, 
                label='Total Species Richness', color='steelblue', edgecolor='black', linewidth=0.7)

# Plot average species richness on the secondary y-axis
bars2 = ax1_twin.bar(x + width/2, sorted_df['Average Species Richness'], width,
                     label='Average Species Richness', color='coral', edgecolor='black', linewidth=0.7)

ax1.set_xticks(x)
ax1.set_xticklabels(sorted_df['Site'], rotation=45, ha='right')
ax1.set_ylabel('Total Species Richness', fontsize=12)
ax1_twin.set_ylabel('Average Species Richness', fontsize=12)
ax1.set_title('Species Richness by Site (Total and Average)', fontsize=14, fontweight='bold')
ax1.tick_params(axis='y')
ax1_twin.tick_params(axis='y')
ax1.grid(axis='y', alpha=0.3)

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper center')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    ax1_twin.text(bar.get_x() + bar.get_width()/2., height,
                  f'{height:.2f}',
                  ha='center', va='bottom', fontsize=8)

# 2. Relationship between Number of Surveys and Species Richness (Scatter Plot)
ax2 = axes[1]
scatter = ax2.scatter(results_df['Number of Surveys'], 
                     results_df['Species Richness'], 
                     s=150, alpha=0.6, c=results_df['Species Richness'], 
                     cmap='viridis', edgecolors='black', linewidth=1)
ax2.set_xlabel('Number of Surveys', fontsize=12)
ax2.set_ylabel('Species Richness', fontsize=12)
ax2.set_title('Species Richness vs Number of Surveys', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.set_xlim(0, results_df['Number of Surveys'].max() + 2.5)

# Add site labels to points
for idx, row in results_df.iterrows():
    ax2.annotate(row['Site'], 
                (row['Number of Surveys'], row['Species Richness']),
                fontsize=9, alpha=0.8, 
                xytext=(5, 5), textcoords='offset points')

# Add colorbar for scatter plot
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Species Richness', rotation=270, labelpad=15)

plt.tight_layout()
plt.show()